**add_messages->is like the operator.add but add messages is more optimized**

In [14]:
from langgraph.graph import StateGraph , START , END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage , HumanMessage
from typing import Annotated , TypedDict
from langgraph.checkpoint.memory import MemorySaver


In [15]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os

load_dotenv()

groq_api = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    groq_api_key=groq_api,
)

In [16]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

In [17]:
graph = StateGraph(ChatState)
checkpointer = MemorySaver()

def chat_node(state: ChatState):
    messages = state["messages"]
    response = llm.invoke(messages)

    return {"messages": [response]}


graph.add_node("chat_node", chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

app = graph.compile(checkpointer=checkpointer)

In [18]:
# response = app.invoke({
#     "messages": [
#         HumanMessage(content="What is langchain?")
#     ]
# })
#
# print(response["messages"][-1].content)

In [19]:
thread_id = '1'
while True:
    user_message = input ("type you  message here:")
    print("User:",user_message)

    if user_message.strip().lower() in ["exit", "quit"]:
        break
    config = {'configurable':{'thread_id':thread_id}}

    response = app.invoke({"messages":[HumanMessage(content=user_message)]},config=config)

    print("AI:",response["messages"][-1].content)

User: hi my name is Yar Khan
AI: Hello Yar Khan, it's nice to meet you. Is there something I can help you with or would you like to chat?
User: What is my name
AI: Your name is Yar Khan.
User: which one is greater 2 or 1
AI: 2 is greater than 1.
User: multiply greater number with 3
AI: The greater number is 2. Multiplying 2 by 3 gives:

2 × 3 = 6
User: exit


## Persistance->You store the state in RAM or database for future use ##

## Thread id is like the ubique token or id that is alloctaed to every user ##